In [ ]:
import torch
from flash_kmeans import batch_kmeans_Euclid
import os

num_layers = 36
n_clusters = 64

# we can combine multiple kv cache data
TENSOR_PATH = "/data/jinda/kv-cache/Qwen3-4B-thinking-2507/"
task_name_list = ["mmlu_pro-10000-tokens", "hendrycks_math-10000-tokens"]

SAVE_PATH = f'{TENSOR_PATH}/gpqa_first_10tasks-80000-tokens/c_{n_clusters}'
os.makedirs(SAVE_PATH, exist_ok=True)

for layer_id in range(num_layers):
    print(f'layer {layer_id} start calibration centroids')

    
    tensor_name = f'kv_calibration_layer_{layer_id}.pt'

    tensor_list = []
    for task_name in task_name_list:
        tensor_list.append(torch.load(TENSOR_PATH + task_name + '/' + tensor_name))

    # concat tensors from different tasks
    full_tensor = {}
    for tensor in tensor_list:
        if 'k' not in full_tensor:
            full_tensor['k'] = tensor['k']
        else:
            full_tensor['k'] = torch.cat([full_tensor['k'], tensor['k']], dim=0)
        if 'v' not in full_tensor:
            full_tensor['v'] = tensor['v']
        else:
            full_tensor['v'] = torch.cat([full_tensor['v'], tensor['v']], dim=0)
        if 'indices' not in full_tensor:
            full_tensor['indices'] = tensor['indices']
        else:
            full_tensor['indices'] = torch.cat([full_tensor['indices'], tensor['indices']], dim=0)


    # TokenSize, HeadSize, DimSize
    T, H, D = full_tensor['k'].shape
    # print(T, H, D)

    # k: 1, TokenSize, HeadSize*DimSize
    k = full_tensor['k'].flatten().view(T, -1)[None, ...].to('cuda:0')
    k_cluster_ids, k_centers, total_iters = batch_kmeans_Euclid(k, n_clusters=n_clusters, max_iters=200,tol=1e-4, verbose=False, use_heuristic=True)
    k_centers = k_centers.squeeze(0)
    torch.save(k_centers, f'{SAVE_PATH}/k_layer_{layer_id}_clusters_{n_clusters}_centers.pt')
    print(f'k_centers saved, shape: {k_centers.shape}, total_iters: {total_iters}')

    v = full_tensor['v'].flatten().view(T, -1)[None, ...].to('cuda:0')
    v_cluster_ids, v_centers, total_iters = batch_kmeans_Euclid(v, n_clusters=n_clusters, max_iters=200,tol=1e-4, verbose=False, use_heuristic=True)
    v_centers = v_centers.squeeze(0)
    torch.save(v_centers, f'{SAVE_PATH}/v_layer_{layer_id}_clusters_{n_clusters}_centers.pt')
    print(f'v_centers saved, shape: {v_centers.shape}, total_iters: {total_iters}')




layer 0 start calibration centroids
torch.Size([30000, 8, 128])
k_centers saved, shape: torch.Size([64, 1024]), total_iters: 37
v_centers saved, shape: torch.Size([64, 1024]), total_iters: 21
layer 1 start calibration centroids
torch.Size([30000, 8, 128])
k_centers saved, shape: torch.Size([64, 1024]), total_iters: 43
v_centers saved, shape: torch.Size([64, 1024]), total_iters: 51
layer 2 start calibration centroids
torch.Size([30000, 8, 128])
k_centers saved, shape: torch.Size([64, 1024]), total_iters: 156
v_centers saved, shape: torch.Size([64, 1024]), total_iters: 63
layer 3 start calibration centroids
torch.Size([30000, 8, 128])
k_centers saved, shape: torch.Size([64, 1024]), total_iters: 74
v_centers saved, shape: torch.Size([64, 1024]), total_iters: 38
layer 4 start calibration centroids
torch.Size([30000, 8, 128])
k_centers saved, shape: torch.Size([64, 1024]), total_iters: 103
v_centers saved, shape: torch.Size([64, 1024]), total_iters: 35
layer 5 start calibration centroids
to

: 